# Домашнее задание Нейронные сети

## Описание

**Основная задача:** по небольшой спектрограмме аудиофайла определить какие ноты были сыграны.
**Дополнительно:** ответить на вопросы, выполнить небольшие задания или написать размышления.

### Дано

- MIDI-файлы;
- Soundfonts-файлы для придания звуков MIDI-файлам.

### Результат

Архитектура модели, гиперпараметры обучения и модели, а также веса сохранненые в формате `.pth` или `.pt`.

### Оценивание

Выше оценка если выше Accuracy и F1. Также в зачет идут качественный и чистый код, правильные размышления в задачках и обоснованность решений в целом.

### P.S.

> MIDI-файлы — музыкальные инструкции сохраненные в виде нот, таймингов, высоты тона, инструментах и т.п. Эти файлы не содержат предзаписанного звука, но могут быть синтезированы в звуковую дорожку (используя звуковые шрифты или другие алгоритмы).   

> Гиперпараметры (в машинном обучени) — параметры настраиваемые до обучения и/или определяющие структуру модели (количество слоев, параметры слоев, скорость обучения, время обучения и т.п.).



❌ Далее идет деструктивная операция, удаляет директории `input`, `midi` и `soundfonts`. ❌

In [ ]:
!rm -rf input midi soundfonts

Разархивация файлов работы.

In [ ]:
!unzip -o hse_imsh_homework_2_lite.zip

Установка зависимостей.

In [ ]:
!pip install midirenderer pretty_midi pillow matplotlib pandas

🔒 Чекпоинт 1, если что-то сломалось после этой ячейки, достаточно перезапустить начиная с этой. 🔒

## Настройки

Настройки могут быть изменены для тестирования, но при оценивании будут использованы значения по умолчанию.

In [ ]:
from IPython.display import display
import ipywidgets as widgets

In [ ]:
SOUNDFONT_VARIANT = widgets.RadioButtons(
    options=["Grand_Piano", "Sine Wave"],
    value="Grand_Piano",
)
PITCH_RANGE = widgets.IntRangeSlider(
    value=[40, 80],
    min=0,
    max=127,
    step=1,
    orientation="horizontal",
)
FRAGMENT_LENGTH = widgets.FloatSlider(
    value=1,
    min=0.1,
    max=20,
)

display(widgets.Label(value="Вариант звукового шрифта:"))
display(SOUNDFONT_VARIANT)
display(widgets.Label(value="Диапозон высоты звука:"))
display(PITCH_RANGE)
display(widgets.Label(value="Время фрагмента:"))
display(FRAGMENT_LENGTH)
with_synthetic_midis = False  # пока не трогаем

Ячейку с кодом перезапускать не нужно, достаточно изменять значения в интерфейсе.

## Подготовка данных

Подгрузка зависимостей.

In [ ]:
from hashlib import md5
from pathlib import Path
import warnings
import math

import midirenderer
import pretty_midi
import tqdm.auto as tqdm
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import random

Отключаем назойливые предупреждения библиотеки `pretty_midi`.

In [ ]:
warnings.filterwarnings("ignore", category=RuntimeWarning, module="pretty_midi")

Создаем пути до основных директорий. Используется ООП подход для работы с путями (встроенная библиотека pathlib).

In [ ]:
soundfonts_path = Path("./soundfonts")
midi_path = Path("./midi")

input_path = Path("./input")
input_path.mkdir(exist_ok=True)
input_midi_path = input_path / "midi"
input_midi_path.mkdir(exist_ok=True)
input_samples_path = input_path / "samples"
input_samples_path.mkdir(exist_ok=True)
input_spectrograms_path = input_path / "spectrograms"
input_spectrograms_path.mkdir(exist_ok=True)

Вспомогательные функции:
- `sample_midi` — разделение MIDI-файла на куски;
- `midi_to_wav` — синтез WAV-файла из MIDI-файла;
- `wav_to_spec` — генерация спектрограммы на основе WAV-файла;
- `midi_to_df` — генерация списка нот в формате датафрейма из MIDI-файла.

In [ ]:
def sample_midi(midi_file: Path, sample_length: float, out_path: Path):
    midi = pretty_midi.PrettyMIDI(midi_file)
    total_time = midi.get_end_time()

    part_number = 0
    start_time = 0

    part_number_length = len(str(int(total_time / sample_length)))
    file_name = md5(midi_file.stem.encode()).hexdigest()
    files = []

    while start_time < total_time:
        end_time = min(start_time + sample_length, total_time)

        new_midi = pretty_midi.PrettyMIDI()

        for instrument in midi.instruments:
            if instrument.is_drum:
                continue
            new_instrument = pretty_midi.Instrument(program=instrument.program)

            for note in instrument.notes:
                if start_time <= note.start < end_time or start_time <= note.end < end_time:
                    new_note = pretty_midi.Note(
                        velocity=note.velocity,
                        pitch=note.pitch,
                        start=max(0, note.start - start_time),
                        end=min(sample_length, note.end - start_time),
                    )
                    if new_note.start < new_note.end:
                        new_instrument.notes.append(new_note)

            new_midi.instruments.append(new_instrument)

        start_time += sample_length

        if new_midi.get_end_time() != sample_length or not len(new_midi.instruments):
            continue

        part_number += 1
        output_file = out_path / f"{file_name}_{str(part_number).zfill(part_number_length)}.mid"
        new_midi.write(output_file)
        files.append(output_file)

    return files

In [ ]:
def midi_to_wav(midi_file: Path, soundfont_file: Path, out_path: Path):
    wav_data = midirenderer.render_wave_from(
        soundfont_file.read_bytes(),
        midi_file.read_bytes(),
    )
    output_file = out_path / (midi_file.stem + ".wav")
    output_file.write_bytes(wav_data)

    del wav_data
    return output_file

In [ ]:
def wav_to_spec(wav_file: Path, out_path: Path, scale: str = "linear"):
    y, sr = librosa.load(wav_file, sr=None)

    orig_spec = librosa.amplitude_to_db(np.abs(librosa.stft(y)))

    spec = np.clip(orig_spec, -40, 40)
    spec = 255 - (spec + 40) / 80 * 255
    spec = spec.astype(np.uint8)

    plt.figure(figsize=(spec.shape[1] / 100, spec.shape[0] / 100), dpi=100)
    plt.axis("off")
    librosa.display.specshow(spec, sr=sr, x_axis=None, y_axis=scale, cmap="gray_r")

    output_file = out_path / (wav_file.stem + ".png")
    plt.savefig(output_file, bbox_inches="tight", pad_inches=0, dpi=100, format="png")
    plt.close()

    img = Image.open(output_file)
    img = img.convert("L")
    img.save(output_file, optimize=True, format="PNG")
    img.close()

    del y, sr, orig_spec, spec, img
    return output_file

In [ ]:
def midi_to_df(midi_file: Path):
    midi = pretty_midi.PrettyMIDI(midi_file)
    notes_data = []

    for instrument in midi.instruments:
        if instrument.is_drum:
            continue
        for note in instrument.notes:
            notes_data.append({
                "file": midi_file.stem,
                "pitch": note.pitch,
                "start": note.start,
                "end": note.end,
                "velocity": note.velocity,
            })

    return pd.DataFrame(notes_data)

In [ ]:
def generate_synthetic_midi(output_path, min_pitch, max_pitch, count=300, max_length=10):
    random.seed(42)

    output_path.mkdir(parents=True, exist_ok=True)

    one_note = max_pitch - min_pitch + 1
    two_note = (count - one_note) / 4
    three_note = (count - one_note) / 6

    pool = list(range(min_pitch, max_pitch + 1)) * (math.ceil(count / one_note) - 1)

    def write_notes(name, *pitches):
        midi = pretty_midi.PrettyMIDI()
        instrument = pretty_midi.Instrument(program=0)
        instrument.notes.extend([
            pretty_midi.Note(velocity=100, pitch=pitch, start=0, end=max_length)
            for pitch in pitches
        ])
        midi.instruments.append(instrument)
        midi.write(output_path / name)

    for pitch in range(min_pitch, max_pitch + 1):
        write_notes(f"note_{pitch}.mid", pitch)

    random.shuffle(pool)
    for i in range(round(two_note)):
        a, b = pool.pop(), pool.pop()
        write_notes(f"note_{a}_{b}.mid", a, b)

    for i in range(round(three_note)):
        a, b, c = pool.pop(), pool.pop(), pool.pop()
        write_notes(f"note_{a}_{b}_{c}.mid", a, b, c)

🔒 Чекпоинт 2. 🔒

Сэмплирование MIDI-файлов.

In [ ]:
soundfont_file = soundfonts_path / f"{SOUNDFONT_VARIANT.value}.sf2"

if with_synthetic_midis:
    generate_synthetic_midi(
        input_midi_path,
        PITCH_RANGE.value[0],
        PITCH_RANGE.value[1],
        count=30 * (PITCH_RANGE.value[1] - PITCH_RANGE.value[0] + 1),
        max_length=FRAGMENT_LENGTH.value
    )

midi_files = list(midi_path.rglob("*.mid"))

for midi_file in tqdm.tqdm(midi_files):
    sample_midi(midi_file, FRAGMENT_LENGTH.value, input_midi_path)

sampled_midi_files = list(input_midi_path.rglob("*.mid"))

In [ ]:
pd.DataFrame([f.stem for f in sampled_midi_files], columns=["file"])

🔒 Чекпоинт 3. 🔒

Конвертация MIDI-сэмплов в WAV-файлы и получения спектрограм.

> Возможно librosa или какая-то другая библиотека украдет у вас всю оперативку... как никак дорогая штука. Если получится дойти до следующего чекпоинта, то можно смело перезапускать среду, главное чтоб файлы остались.

In [ ]:
dfs = []
for midi_file in tqdm.tqdm(sampled_midi_files):
    df = midi_to_df(midi_file)
    dfs.append(df)
    wav_file = midi_to_wav(midi_file, soundfont_file, input_samples_path)
    wav_to_spec(wav_file, input_spectrograms_path, "log")

In [ ]:
notes_df = pd.concat(dfs, ignore_index=True)
del dfs
notes_df.to_csv(input_path / "notes_metadata.csv", index=False)

In [ ]:
1 / 0

🔒 Чекпоинт 4. 🔒

На текущий момент у вас должен быть файл `notes_metadata.csv` с нотами в каждом сэмпле (MIDI и WAV), а также спектрограммы каждого сэмпла. Можно спокойно перезапускать или останавливать среду, а потом продложить с этого места. Достаточно выполнить ячейки между 1 и 2 чекпоинтами (импорт, полезные функции, переменные путей)

In [ ]:
notes_df = pd.read_csv(input_path / "notes_metadata.csv")
notes_df

❓ Задание 1. ❓

Проанализируйте notes_df и определить возможные проблемные точки в обучении модели, например, распределения данных.

> ЗДЕСЬ ИДЕТ ВАШ ОТВЕТ (двойной клик для редактирования)

In [ ]:
# ЗДЕСЬ ИДЕТ ВАШ КОД

Список файлов спектрограм подходящих под условия задачи.

In [ ]:
in_range_specs_df = notes_df.groupby("file")["pitch"].apply(list).reset_index()
in_range_specs_df = in_range_specs_df[
    in_range_specs_df["pitch"].apply(lambda pitches: all(PITCH_RANGE.value[0] <= pitch <= PITCH_RANGE.value[1] for pitch in pitches))
]
in_range_specs_df["train"] = in_range_specs_df["file"].apply(lambda file: file.startswith("note_"))
in_range_specs_df

In [ ]:
false_train_indices = in_range_specs_df[in_range_specs_df['train'] == False].index

if not false_train_indices.empty:
    indices_to_set_true, _ = train_test_split(
        false_train_indices,
        test_size=0.2,
        random_state=42,
        shuffle=True
    )

    in_range_specs_df.loc[indices_to_set_true, 'train'] = True

## Подготовка train/test данных

🔒 Чекпоинт 4. 🔒

In [ ]:
num_classes = PITCH_RANGE.value[1] - PITCH_RANGE.value[0] + 1

train_df = in_range_specs_df[in_range_specs_df['train'] == True].reset_index(drop=True)
test_df = in_range_specs_df[in_range_specs_df['train'] == False].reset_index(drop=True)

X_train_list = []
Y_train_list = []
X_test_list = []
Y_test_list = []

image_size = None

for i, row in tqdm.tqdm(train_df.iterrows(), total=len(train_df), desc="Загрузка изображений для обучения"):
    image_path = row["file"]
    img = Image.open(input_spectrograms_path / f"{image_path}.png")
    if img.mode != 'L':
        img = img.convert('L')

    if image_size is None:
        image_size = img.size

    assert img.size == image_size, f"Внимание: Изображение {image_path}.png имеет другой размер {img.size}, ожидается {image_size}."


    img_array = np.array(img, dtype=np.float32) / 255.0
    X_train_list.append(img_array)

    pitch_values = row["pitch"]
    if not isinstance(pitch_values, list):
        try:
            import ast
            pitch_values = ast.literal_eval(pitch_values)
        except (ValueError, SyntaxError) as e:
            print(f"Внимание: Невозможно обработать значения pitch для {image_path}: {pitch_values}")
            raise e

    one_hot_label = np.zeros(num_classes, dtype=np.float32)
    for value in pitch_values:
        assert PITCH_RANGE.value[0] <= value <= PITCH_RANGE.value[1], f"Внимание: Значение pitch {value} вне диапазона {PITCH_RANGE.value} для файла {image_path}"
        one_hot_label[value - PITCH_RANGE.value[0]] = 1
    Y_train_list.append(one_hot_label)

for i, row in tqdm.tqdm(test_df.iterrows(), total=len(test_df), desc="Загрузка изображений для теста"):
    image_path = row["file"]
    img = Image.open(input_spectrograms_path / f"{image_path}.png")
    if img.mode != 'L':
        img = img.convert('L')

    assert img.size == image_size, f"Внимание: Изображение {image_path}.png имеет другой размер {img.size}, ожидается {image_size}."

    img_array = np.array(img, dtype=np.float32) / 255.0
    X_test_list.append(img_array)

    pitch_values = row["pitch"]
    if not isinstance(pitch_values, list):
        try:
            import ast
            pitch_values = ast.literal_eval(pitch_values)
        except (ValueError, SyntaxError) as e:
            print(f"Внимание: Невозможно обработать значения pitch для {image_path}: {pitch_values}")
            raise e

    one_hot_label = np.zeros(num_classes, dtype=np.float32)
    for value in pitch_values:
        assert PITCH_RANGE.value[0] <= value <= PITCH_RANGE.value[1], f"Внимание: Значение pitch {value} вне диапазона {PITCH_RANGE.value} для файла {image_path}"
        one_hot_label[value - PITCH_RANGE.value[0]] = 1
    Y_test_list.append(one_hot_label)

X_train = np.array(X_train_list)
Y_train = np.array(Y_train_list)
X_test = np.array(X_test_list)
Y_test = np.array(Y_test_list)

In [ ]:
print("Форма X_train:", X_train.shape)
print("Форма Y_train:", Y_train.shape)
print("Форма X_test:", X_test.shape)
print("Форма Y_test:", Y_test.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 8))

axes[0].imshow(X_train[0], cmap="gray", aspect="auto")
axes[0].set_title("Пример X_train")

axes[1].imshow(X_test[0], cmap="gray", aspect="auto")
axes[1].set_title("Пример X_test")

plt.tight_layout()
plt.show()

In [ ]:
print("\nПример Y_train (one-hot encoded):")
print(Y_train[:5])

print("\nПример Y_test (one-hot encoded):")
print(Y_test[:5])

In [ ]:
X_train_tensor = torch.from_numpy(X_train).type(torch.float32)
X_test_tensor = torch.from_numpy(X_test).type(torch.float32)
Y_train_tensor = torch.from_numpy(Y_train).type(torch.float32)
Y_test_tensor = torch.from_numpy(Y_test).type(torch.float32)

print(f"Размеры тензоров:")
print(f"X_train: {X_train_tensor.shape}")
print(f"Y_train: {Y_train_tensor.shape}")
print(f"X_test: {X_test_tensor.shape}")
print(f"Y_test: {Y_test_tensor.shape}")

## Построение модели

🔒 Чекпоинт 5. 🔒

In [ ]:
def calculate_multi_label_metrics(y_true, y_pred_logits, threshold=0.5):
    y_pred_probs = torch.sigmoid(torch.from_numpy(y_pred_logits)).numpy()
    y_pred_binary = (y_pred_probs >= threshold).astype(int)
    accuracy = accuracy_score(y_true, y_pred_binary)
    precision = precision_score(y_true, y_pred_binary, average="samples", zero_division=0)
    recall = recall_score(y_true, y_pred_binary, average="samples", zero_division=0)
    f1 = f1_score(y_true, y_pred_binary, average="samples", zero_division=0)
    return accuracy, precision, recall, f1

In [ ]:
random_seed = 42

random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed);

def init_normal(m):
    if hasattr(m, "weight"):
        nn.init.xavier_uniform_(m.weight)

In [ ]:
train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

batch_size = 64
num_workers = 0

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

## Полет фантазий

In [ ]:
class MyModel(nn.Module):
    def __init__(self, input_dims: tuple[int, int], num_classes: int):
        super(MyModel, self).__init__()
        # ЗДЕСЬ ИДЕТ ВАШ КОД

    def forward(self, x):
        pass  # нужно убрать, если вы что-то написали в функции
        # ЗДЕСЬ ИДЕТ ВАШ КОД

In [ ]:
model = MyModel((X_train.shape[1], X_train.shape[2]), Y_train.shape[1])
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

model.apply(init_normal)  # опционально

print("Архитектура модели:")
print(model)
print("\nОшибка:")
print(criterion)
print("\nОптимизатор:")
print(optimizer)

In [ ]:
num_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"\nИспользуется устройство: {device}")

train_losses = []
train_metrics = []
val_losses = []
val_metrics = []

print("\nНачало обучения (Multi-Label)...")

for epoch in range(num_epochs):
    model.train()
    running_loss_train = 0.0
    train_y_true_all = []
    train_y_pred_logits_all = []

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss_train += loss.item()
        train_y_true_all.append(labels.cpu().numpy())
        train_y_pred_logits_all.append(outputs.cpu().detach().numpy())

    epoch_loss_train = running_loss_train / len(train_loader)
    train_y_true_np = np.concatenate(train_y_true_all)
    train_y_pred_logits_np = np.concatenate(train_y_pred_logits_all)

    train_losses.append(epoch_loss_train)
    train_metrics.append(calculate_multi_label_metrics(train_y_true_np, train_y_pred_logits_np))

    model.eval()
    running_loss_val = 0.0
    val_y_true_all = []
    val_y_pred_logits_all = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss_val += loss.item()
            val_y_true_all.append(labels.cpu().numpy())
            val_y_pred_logits_all.append(outputs.cpu().numpy())

    epoch_loss_val = running_loss_val / len(test_loader)
    val_y_true_np = np.concatenate(val_y_true_all)
    val_y_pred_logits_np = np.concatenate(val_y_pred_logits_all)

    val_losses.append(epoch_loss_val)
    val_metrics.append(calculate_multi_label_metrics(val_y_true_np, val_y_pred_logits_np))

    print(f"Epoch [{epoch+1}/{num_epochs}], "
          f"Train Loss: {epoch_loss_train:.4f}, Train F1: {train_metrics[-1][3]:.4f}, "
          f"Val Loss: {epoch_loss_val:.4f}, Val F1: {val_metrics[-1][3]:.4f}")

print("\nОбучение завершено!")

print("\nФинальная оценка на тестовом наборе (Multi-Label):")
final_accuracy, final_precision, final_recall, final_f1 = calculate_multi_label_metrics(
    np.concatenate(val_y_true_all),
    np.concatenate(val_y_pred_logits_all)
)
print(f"Test Accuracy (samples): {final_accuracy:.4f}")
print(f"Test Precision (samples): {final_precision:.4f}")
print(f"Test Recall (samples): {final_recall:.4f}")
print(f"Test F1-Score (samples): {final_f1:.4f}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16, 5))

plt.subplot(1, 3, 1)
plt.plot(range(1, len(train_metrics) + 1), train_losses, label='Train Loss')
plt.plot(range(1, len(val_metrics) + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss over Epochs')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(range(1, len(train_metrics) + 1), list(zip(*train_metrics))[0], label='Train Accuracy')
plt.plot(range(1, len(val_metrics) + 1), list(zip(*val_metrics))[0], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(range(1, len(train_metrics) + 1), list(zip(*train_metrics))[3], label='Train F1')
plt.plot(range(1, len(val_metrics) + 1), list(zip(*val_metrics))[3], label='Validation F1')
plt.xlabel('Epoch')
plt.ylabel('F1')
plt.title('F1 over Epochs')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

❓ Задание 2. ❓

Проанализируйте результаты обучения, распишите текстом и если надо в виде кода ваши размышления.

> ЗДЕСЬ ИДЕТ ВАШ ОТВЕТ

In [ ]:
# ЗДЕСЬ ИДЕТ ВАШ КОД

❓ Задание 3. ❓

Изучите документацию Pytorch или другие источники и экспортируйте веса модели.

> ЗДЕСЬ ИДЕТ ВАШ ОТВЕТ

In [ ]:
# ЗДЕСЬ ИДЕТ ВАШ КОД

❓ Задание 4. ❓

Измените значение в ячейке с настройками с `with_synthetic_midis = False` на `with_synthetic_midis = True` (либо выполните код ниже). Вернитесь к чекпоинту 1. Проанализируйте изменения в результатах задания 1 с синтетическими данными и без. Проанализируйте изменения в качестве модели.


> ЗДЕСЬ ИДЕТ ВАШ ОТВЕТ

In [ ]:
# ЗДЕСЬ ИДЕТ ВАШ КОД

In [ ]:
with_synthetic_midis = True

❓ Задание 5. ❓

Сделайте выводы по проделанной работе. С какими проблемами столкнулись и как их разрешили.

> ЗДЕСЬ ИДЕТ ВАШ ОТВЕТ

In [ ]:
# ЗДЕСЬ ИДЕТ ВАШ КОД